# Plotting Final — Economic Series Movement Around May 1995

Goal: inspect **how all economic series move during the same calendar year as the reference event, May 1995**. Each series is shown in a shared event window from January 1995 through December 1995, with May 1995 marked clearly. The method is applied consistently to every CSV in `data/cleaned`.

For each series, the notebook plots the observed values during 1995 and the 2- and 4-observation rolling means, plus a standardized view using that series' full-history mean and standard deviation for context. Series with no observations in 1995 are reported as unavailable for that year's movement view rather than silently substituted. Observation frequency may differ between series, so 2 and 4 rolling observations do not always equal 2 and 4 months.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ---- Shared observation-year controls ----
TARGET_YEAR = 1995
TARGET_MONTH = 5
ROLLING_WINDOWS = (2, 4)

# Event window: the whole calendar year containing the reference month.
WINDOW_START = pd.Timestamp(TARGET_YEAR, 1, 1)
WINDOW_END = pd.Timestamp(TARGET_YEAR, 12, 31)
TARGET_START = pd.Timestamp(TARGET_YEAR, TARGET_MONTH, 1)
TARGET_END = TARGET_START + pd.offsets.MonthEnd(1)

# Locate data regardless of whether Jupyter starts in repository root or notebook folder.
candidates = [Path('data/cleaned'), Path('../data/cleaned'), Path('../../data/cleaned')]
cleaned_dir = next((p.resolve() for p in candidates if p.exists()), None)
if cleaned_dir is None:
    raise FileNotFoundError('Cannot find data/cleaned. Start Jupyter from repository root or notebook folder.')

csv_paths = sorted(cleaned_dir.glob('*.csv'))
if not csv_paths:
    raise FileNotFoundError(f'No CSV files found in {cleaned_dir}')

print(f'Common observation year: {TARGET_YEAR}')
print(f'Event month highlighted: {TARGET_START:%B %Y}')
print(f'Window shown: {WINDOW_START:%Y-%m-%d} to {WINDOW_END:%Y-%m-%d}')
print(f'Rolling windows for every series: {ROLLING_WINDOWS} observations')
print(f'CSV files discovered: {len(csv_paths)}')

series_by_id = {}
load_issues = []
for csv_path in csv_paths:
    series_id = csv_path.stem
    try:
        raw = pd.read_csv(csv_path)
        if not {'date', 'value'}.issubset(raw.columns):
            load_issues.append((series_id, f"Missing date/value columns: {list(raw.columns)}"))
            continue
        raw['date'] = pd.to_datetime(raw['date'], errors='coerce')
        raw['value'] = pd.to_numeric(raw['value'], errors='coerce')
        s = (raw.dropna(subset=['date', 'value'])
                 .sort_values('date')
                 .drop_duplicates('date', keep='last')
                 .set_index('date')['value'])
        if s.empty:
            load_issues.append((series_id, 'No valid date/value observations'))
            continue
        series_by_id[series_id] = s
    except Exception as exc:
        load_issues.append((series_id, str(exc)))

if not series_by_id:
    raise ValueError('No valid economic series could be loaded.')

print(f'Valid series loaded: {len(series_by_id)}')
if load_issues:
    print(f'Series skipped due to data/schema issues: {len(load_issues)}')
    display(pd.DataFrame(load_issues, columns=['series_id', 'issue']))

# Summary explicitly checks data availability during the target year and month.
summary_rows = []
for series_id, s in sorted(series_by_id.items()):
    year_values = s.loc[(s.index >= WINDOW_START) & (s.index <= WINDOW_END)]
    may_values = s.loc[(s.index >= TARGET_START) & (s.index <= TARGET_END)]
    mu, sigma = s.mean(), s.std(ddof=1)
    summary_rows.append({
        'series_id': series_id,
        'first_date': s.index.min(),
        'last_date': s.index.max(),
        'observations_in_1995': len(year_values),
        'observations_in_may_1995': len(may_values),
        'may_1995_value_first_obs': may_values.iloc[0] if len(may_values) else np.nan,
        '1995_value_first_obs': year_values.iloc[0] if len(year_values) else np.nan,
        '1995_value_last_obs': year_values.iloc[-1] if len(year_values) else np.nan,
        'full_history_mean': mu,
        'full_history_sample_std': sigma
    })

summary = pd.DataFrame(summary_rows).sort_values('series_id').reset_index(drop=True)
display(summary)


## All-series movement during calendar year 1995

Each chart is zoomed to January–December 1995 so you can inspect the movement through that year, rather than looking at the entire multi-decade history. May is highlighted as the shared reference month. The standardized panel provides historical context, while the raw panel preserves the series' original units.

In [ ]:
# Apply the same 1995 event-window view to every economic series.
for series_id, s in sorted(series_by_id.items()):
    year_s = s.loc[(s.index >= WINDOW_START) & (s.index <= WINDOW_END)]
    if year_s.empty:
        print(f'{series_id}: no observations in {TARGET_YEAR}; cannot plot its movement during that year.')
        continue

    full_mean = s.mean()
    full_std = s.std(ddof=1)
    year_z = ((year_s - full_mean) / full_std) if pd.notna(full_std) and full_std != 0 else None
    rolling_2 = year_s.rolling(window=ROLLING_WINDOWS[0], min_periods=1).mean()
    rolling_4 = year_s.rolling(window=ROLLING_WINDOWS[1], min_periods=1).mean()
    may_mask = (year_s.index >= TARGET_START) & (year_s.index <= TARGET_END)

    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, constrained_layout=True)
    for ax in axes:
        ax.axvspan(TARGET_START, TARGET_END, color='orange', alpha=0.22,
                   label=f'Reference month: {TARGET_START:%B %Y}')

    if year_z is not None:
        axes[0].plot(year_z.index, year_z.values, marker='o', markersize=3,
                     linewidth=1.2, label='Z-score (full-history baseline)')
        axes[0].axhline(0, color='black', linewidth=1, label='Full-history mean')
        axes[0].axhline(1, linestyle='--', alpha=0.5, label='+/- 1 std')
        axes[0].axhline(-1, linestyle='--', alpha=0.5)
        if may_mask.any():
            axes[0].scatter(year_z.loc[may_mask].index, year_z.loc[may_mask].values,
                            color='darkorange', edgecolor='black', zorder=5, label='May observation(s)')
        axes[0].set_ylabel('Z-score')
        axes[0].legend(fontsize=8, ncol=2)
    else:
        axes[0].text(0.5, 0.5, 'Z-score unavailable: insufficient variation',
                     transform=axes[0].transAxes, ha='center', va='center')
        axes[0].set_ylabel('Z-score')

    axes[0].set_title(f'{series_id} — movement during {TARGET_YEAR} (standardized)')
    axes[0].grid(True, alpha=0.25)

    axes[1].plot(year_s.index, year_s.values, marker='o', markersize=3,
                 linewidth=1.2, label='Observed value')
    axes[1].plot(year_s.index, rolling_2.values, linewidth=1.2,
                 label='Rolling mean (2 observations)')
    axes[1].plot(year_s.index, rolling_4.values, linewidth=1.5,
                 label='Rolling mean (4 observations)')
    if may_mask.any():
        axes[1].scatter(year_s.loc[may_mask].index, year_s.loc[may_mask].values,
                        color='darkorange', edgecolor='black', zorder=5, label='May observation(s)')
    axes[1].set_title(f'Raw {series_id} values during {TARGET_YEAR}')
    axes[1].set_ylabel('Original units')
    axes[1].set_xlabel('Date')
    axes[1].legend(fontsize=8, ncol=2)
    axes[1].grid(True, alpha=0.25)
    plt.show()

    print(f'{series_id}: {len(year_s)} observation(s) in {TARGET_YEAR}; '
          f'{int(may_mask.sum())} observation(s) in May.')
    display(year_s.rename('value').to_frame())


## Interpretation checklist

- Follow each series from the start to the end of 1995 and note whether it rises, falls, turns, or remains relatively stable.
- Compare movements around May 1995 across series using the standardized panel; use the raw panel for each series' original units.
- Confirm whether each series actually has a May observation. Monthly, quarterly, and irregular series may show different numbers of points.
- Rolling means are calculated from observations within the displayed 1995 window; the first few values therefore use fewer observations when `min_periods=1`.
- The standardized baseline uses the full available history, including data after 1995. This is suitable for retrospective visualization, not a no-look-ahead trading test.
- Similar-looking movements do not prove correlation, causality, or mean reversion; those require separate statistical tests.

## Benchmark-centered spread and 4-observation ROC-weighted spread

The benchmark comes from the top summary table: each series' full-history mean and sample standard deviation. Centering uses `X_centered = X - full_history_mean`, so the historical mean maps to zero. The final spread is the last available observation in May–December 1995 expressed relative to that benchmark and scaled by the series' standard deviation: `spread_z = (last_value - full_history_mean) / full_history_sample_std`. Thus, spread is measured in standard-deviation units and is comparable across series. The second metric multiplies this benchmark-relative spread by the 4-observation backward ROC, `(X_t / X_(t-4)) - 1`, evaluated at the horizon endpoint.

Series with missing/zero standard deviation or undefined ROC are shown as unavailable in the raw table; for the exploratory polynomial plots only, unavailable fit inputs are filled with 0 and clearly labeled. The benchmark mean/std use each series' full history, including observations after 1995, so this is retrospective analysis rather than a no-look-ahead forecast.

In [ ]:
# Benchmark-relative compact table: use full-history mean and sample std from the top summary table.
# Spread is the final observation's standardized distance from the mean benchmark (mean -> 0).
# The 4-observation backward ROC is applied later in the polynomial section.

DERIVATIVE_START = pd.Timestamp(1995, 5, 1)
DERIVATIVE_END = pd.Timestamp(1995, 12, 31)

compact_rows = []

for series_id, s in sorted(series_by_id.items()):
    horizon = s.loc[(s.index >= DERIVATIVE_START) & (s.index <= DERIVATIVE_END)].copy()
    full_mean = s.mean()
    full_std = s.std(ddof=1)

    if horizon.empty:
        compact_rows.append({
            'data_name': series_id,
            'benchmark_mean_full_history': full_mean,
            'benchmark_std_full_history': full_std,
            'last_data_date': pd.NaT,
            'last_data_value': np.nan,
            'observations_in_horizon': 0,
            'centered_last_value': np.nan,
            'spread_zscore_from_zero_benchmark': np.nan,
            'status': 'No observations in May-Dec 1995'
        })
        continue

    last_date = horizon.index[-1]
    last_value = float(horizon.iloc[-1])
    centered_last = last_value - full_mean
    spread_z = centered_last / full_std if pd.notna(full_std) and full_std != 0 else np.nan
    status = 'OK' if pd.notna(spread_z) else 'Spread unavailable: std is zero or undefined'

    compact_rows.append({
        'data_name': series_id,
        'benchmark_mean_full_history': full_mean,
        'benchmark_std_full_history': full_std,
        'last_data_date': last_date,
        'last_data_value': last_value,
        'observations_in_horizon': len(horizon),
        'centered_last_value': centered_last,
        'spread_zscore_from_zero_benchmark': spread_z,
        'status': status
    })

compact_derivative_spread = pd.DataFrame(compact_rows).sort_values('data_name').reset_index(drop=True)
display(compact_derivative_spread)


## Polynomial fit: benchmark-relative spread and spread × 4-observation backward ROC

The spread is the last observation's standardized distance from the full-history mean benchmark: `(X_t - mean_full_history) / std_full_history`. Since the series is centered, the mean benchmark is zero; dividing by standard deviation expresses distance in standard-deviation units. The second metric multiplies this spread by the original series' 4-observation backward ROC, `(X_t / X_(t-4)) - 1`, measured at the last observation in the May–December 1995 horizon.

The x-axis is the alphabetical series index, so these are cross-series comparison curves, not time-series fits. ROC is a relative change, not a calculus derivative; the 4-observation lag spans different calendar durations across series. For exploratory polynomial fitting only, unavailable values are filled with 0.

In [ ]:
# Compare benchmark-relative standardized spread with spread weighted by 4-observation backward ROC.
# Missing values are preserved in the displayed table; NaN fit inputs are set to 0 for exploratory plots.

POLYNOMIAL_DEGREE = 2
ROC_LAG_OBSERVATIONS = 4

poly_data = compact_derivative_spread.copy().sort_values('data_name').reset_index(drop=True)
poly_data['spread_for_fit'] = pd.to_numeric(
    poly_data['spread_zscore_from_zero_benchmark'], errors='coerce'
)

roc_values = []
roc_dates = []
roc_lag_dates = []
for series_id in poly_data['data_name']:
    s = series_by_id[series_id].sort_index()
    horizon = s.loc[(s.index >= DERIVATIVE_START) & (s.index <= DERIVATIVE_END)]
    if horizon.empty:
        roc_values.append(np.nan)
        roc_dates.append(pd.NaT)
        roc_lag_dates.append(pd.NaT)
        continue

    end_date = horizon.index[-1]
    end_position = s.index.get_indexer([end_date])[0]
    lag_position = end_position - ROC_LAG_OBSERVATIONS
    if lag_position < 0:
        roc_values.append(np.nan)
        roc_dates.append(end_date)
        roc_lag_dates.append(pd.NaT)
        continue

    x_now = float(s.iloc[end_position])
    x_lag = float(s.iloc[lag_position])
    if x_lag == 0 or not np.isfinite(x_lag):
        roc_values.append(np.nan)
    else:
        roc_values.append((x_now / x_lag) - 1.0)
    roc_dates.append(end_date)
    roc_lag_dates.append(s.index[lag_position])

poly_data['roc_4_observation'] = roc_values
poly_data['roc_end_date'] = roc_dates
poly_data['roc_lag_date'] = roc_lag_dates
poly_data['roc_for_fit'] = pd.to_numeric(poly_data['roc_4_observation'], errors='coerce')
poly_data['spread_times_roc_4'] = poly_data['spread_for_fit'] * poly_data['roc_for_fit']

# Fill only the polynomial fitting vectors; keep NaNs in the audit/display columns.
poly_data['spread_for_fit_plot'] = poly_data['spread_for_fit'].fillna(0.0)
poly_data['spread_times_roc_4_plot'] = poly_data['spread_times_roc_4'].fillna(0.0)
poly_data['series_index'] = np.arange(len(poly_data), dtype=float)

if len(poly_data) <= POLYNOMIAL_DEGREE:
    raise ValueError(f'Need more than {POLYNOMIAL_DEGREE} series to fit degree {POLYNOMIAL_DEGREE}.')

x_smooth = np.linspace(poly_data['series_index'].min(), poly_data['series_index'].max(), 500)
spread_coefficients = np.polyfit(poly_data['series_index'], poly_data['spread_for_fit_plot'], POLYNOMIAL_DEGREE)
spread_polynomial = np.poly1d(spread_coefficients)

plt.figure(figsize=(14, 6))
plt.scatter(poly_data['series_index'], poly_data['spread_for_fit_plot'], alpha=0.75,
            label='Benchmark-relative spread in std units (NaN → 0 for fit)')
plt.plot(x_smooth, spread_polynomial(x_smooth), linewidth=2,
         label=f'Polynomial fit, degree {POLYNOMIAL_DEGREE}')
plt.axhline(0, color='black', linewidth=1, alpha=0.7, label='Mean benchmark = 0')
plt.title('Polynomial fit of benchmark-relative spread — May–December 1995')
plt.xlabel('Economic series index (alphabetical order)')
plt.ylabel('Spread (standard deviations from benchmark)')
plt.grid(True, alpha=0.25)
plt.legend()
plt.show()

weighted_coefficients = np.polyfit(poly_data['series_index'], poly_data['spread_times_roc_4_plot'], POLYNOMIAL_DEGREE)
weighted_polynomial = np.poly1d(weighted_coefficients)
plt.figure(figsize=(14, 6))
plt.scatter(poly_data['series_index'], poly_data['spread_times_roc_4_plot'], alpha=0.75,
            label='Spread × 4-observation ROC (NaN → 0 for fit)')
plt.plot(x_smooth, weighted_polynomial(x_smooth), linewidth=2,
         label=f'Polynomial fit, degree {POLYNOMIAL_DEGREE}')
plt.axhline(0, color='black', linewidth=1, alpha=0.7, label='Mean benchmark = 0')
plt.title('Polynomial fit of benchmark-relative spread × 4-observation ROC — May–December 1995')
plt.xlabel('Economic series index (alphabetical order)')
plt.ylabel('Spread (std units) × ROC')
plt.grid(True, alpha=0.25)
plt.legend()
plt.show()

print('Series included in polynomial plots; spread is benchmark-relative and ROC is measured at the horizon endpoint:')
display(poly_data[[
    'data_name', 'series_index', 'benchmark_mean_full_history', 'benchmark_std_full_history',
    'last_data_date', 'last_data_value', 'centered_last_value',
    'spread_zscore_from_zero_benchmark', 'roc_lag_date', 'roc_end_date',
    'roc_4_observation', 'spread_times_roc_4'
]].merge(
    compact_derivative_spread[['data_name', 'benchmark_mean_full_history', 'benchmark_std_full_history']],
    on='data_name', how='left', suffixes=('', '_table')
).drop(columns=['benchmark_mean_full_history_table', 'benchmark_std_full_history_table']))
